# Research Question 2

**Which pathologies does the model treat similarly?**

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.neighbors import NearestNeighbors

In [2]:
# Import data from the base notebook
import import_ipynb
from IPython.utils.capture import capture_output

In [3]:
# Import data from the base notebook

# Use "capture" do suppress plotting images
with capture_output() as captured:
    %run r0_dataset_overiew_dashboard.ipynb

In [4]:
HTML_OUTPUT_FOLDER = "html_plots/r2"

## 1 Compute correlations of model's predictions and show in heatmap

In [5]:
pathology_names = XRAY_DATASET.pathologies
predictions_df = pd.DataFrame(
    MODEL_SCORE_PREDICTIONS_CONVERTED,
    columns=DATASET_PATHOLOGIES
)

correlation_matrix = predictions_df.corr(method="pearson")

In [6]:
def plot_similarity_as_heatmap(correlation_matrix):
    fig = px.imshow(
        correlation_matrix,
        x=pathology_names,
        y=pathology_names,
        text_auto=".2f",
        color_continuous_scale="RdBu_r",
        zmin=-1,
        zmax=1,
        aspect="auto",
        title="Correlation Between Pathology Predictions"
    )

    fig.update_layout(
        xaxis_title="Pathology",
        yaxis_title="Pathology",
        # Make the heatmap less weird shaped!
        width=1100,
        height=900,
        xaxis_tickangle=-45
    )


    fig.show()

In [7]:
plot_similarity_as_heatmap(correlation_matrix)

### Observation

The heatmap shows  that some pathologies are treated similarly. For example, Atelictrasis and Lung Opacity seem often predicted together. Its seems that there is a small group that is often predicted together, containing Lung Opacity, Effusion, Atelectrasis, Consolidation and Edema.

## 2 Compute correlations of ground-truth and show in heatmap

In [8]:
ground_truth_df = pd.DataFrame(
    XRAY_LABELS_LIST,
    columns=DATASET_PATHOLOGIES
)

correlation_matrix_ground_truth = ground_truth_df.corr(method="pearson")

In [9]:
plot_similarity_as_heatmap(correlation_matrix_ground_truth)

### Observation

The correlation between the ground-truth pathology labels is weaker than for the model's prediction score, but it also shows that some pathologies actually occur more often together which is also correctly identified by the model. For exmaple, the group with Lung Opacity, Effusion, Atelectrasis, Consolidation and Edema also forms a cluster in the ground-truth heatmap, altought there correlation is much weaker. 

One interpretation: The model correctly classifies a group of pathologies that occur often together, but fails to distinguish them.

## 3 Compute difference correlations and show in heatmap

In [10]:
correlation_difference = (
    correlation_matrix - correlation_matrix_ground_truth
)

fig = px.imshow(
    correlation_difference,
    x=pathology_names,
    y=pathology_names,
    text_auto=".2f",
    color_continuous_scale="RdBu_r",
    zmin=-1,
    zmax=1,
    aspect="auto",
    title="Difference: Model Correlation vs. Ground Truth"
)

fig.show()

### Observation

The difference shows that the model's predictions are more correlated than the ground-truth, since all values ( except diagonal) are positive.

A strong correlation that only occurs in the model's prediction is Pneumonia, which has several cells with high positive values.

## 4 Investigate high correlations with scatter plots

In [11]:
def show_correlation_scatter_plot(path1, path2, ground_truth_df, predictions_df, threshold=THRESHOLD):

    scatter_df = pd.DataFrame({
        f"{path1} Score": predictions_df[path1],
        f"{path2} Score": predictions_df[path2],
        path1: ground_truth_df[path1].astype(int),
        path2: ground_truth_df[path2].astype(int),
    })

    scatter_df["Image Index"] = scatter_df.index

    conditions = [
        (scatter_df[path1] == 0) & (scatter_df[path2] == 0),
        (scatter_df[path1] == 1) & (scatter_df[path2] == 0),
        (scatter_df[path1] == 0) & (scatter_df[path2] == 1),
        (scatter_df[path1] == 1) & (scatter_df[path2] == 1)
    ]

    categories = [
        "Neither pathology",
        f"{path1} only",
        f"{path2} only",
        "Both pathologies"
    ]

    scatter_df["Ground Truth Combination"] = pd.Series(
        pd.Categorical(
            pd.Series(
                np.select(conditions, categories, default="Unknown"),
                index=scatter_df.index
            ),
            categories=categories
        )
    )

    symbol_map = {
        "Neither pathology": "circle",
        f"{path1} only": "square",
        f"{path2} only": "triangle-up",
        "Both pathologies": "diamond"
    }

    color_map = {
        "Neither pathology": "#636EFA",
        f"{path1} only": "#EF553B",
        f"{path2} only": "#00CC96",
        "Both pathologies": "#AB63FA"
    }

    fig = px.scatter(
        scatter_df,
        x=f"{path1} Score",
        y=f"{path2} Score",
        color="Ground Truth Combination",
        symbol="Ground Truth Combination",
        symbol_map=symbol_map,
        color_discrete_map=color_map,
        hover_data=["Image Index"],
        category_orders={"Ground Truth Combination": categories},
        title=f"Model Predictions: {path1} vs. {path2}",
        labels={
            f"{path1} Score": f"{path1} Prediction Score",
            f"{path2} Score": f"{path2} Prediction Score",
            "Ground Truth Combination": "Actual Labels"
        }
    )

    fig.update_traces(
        marker=dict(size=7, opacity=0.7)
    )

    fig.add_hline(
        y=threshold,
        line_dash="dash",
        line_color="black",
        annotation_text=f"{path2} threshold"
    )

    fig.add_vline(
        x=threshold,
        line_dash="dash",
        line_color="black",
        annotation_text=f"{path1} threshold"
    )

    fig.update_layout(
        width=1100,
        height=800,
        legend_title="Ground Truth",
        template="plotly_white"
    )

    fig.show()

In [12]:
show_correlation_scatter_plot("Lung Lesion", "Pneumonia", ground_truth_df, predictions_df)

### Observation

Lung Lesion and Pneumonia increase indeed together, altough the relationship is not perfectly linear. There are a lot of false positives in the right top area. 

The model may respond to image patterns that influence both pathology scores.

Note: The AI suggest to focus on Lung Lesion and Pneumonia for further visualizations.

In [16]:
both_fp = (
    (pred_ll == 1) &
    (pred_pneu == 1) &
    (true_ll == 0) &
    (true_pneu == 0)
)

fp_df = pd.DataFrame({
    "Lung Lesion Score": predictions_df["Lung Lesion"],
    "Pneumonia Score": predictions_df["Pneumonia"],
    "Error Type": both_fp.map({
        True: "Both false positives",
        False: "Other images"
    })
})

fig = px.scatter(
    fp_df,
    x="Lung Lesion Score",
    y="Pneumonia Score",
    color="Error Type",
    color_discrete_map={
        "Both false positives": "red",
        "Other images": "lightgrey"
    },
    title="Images with Simultaneous False Positives",
    opacity=0.7
)

fig.add_hline(y=THRESHOLD, line_dash="dash", line_color="black")
fig.add_vline(x=THRESHOLD, line_dash="dash", line_color="black")

fig.update_layout(width=1000, height=750)

fig.show()

In [15]:
pred_ll = (predictions_df["Lung Lesion"] >= THRESHOLD).astype(int)
pred_pneu = (predictions_df["Pneumonia"] >= THRESHOLD).astype(int)


true_ll = ground_truth_df["Lung Lesion"].astype(int)
true_pneu = ground_truth_df["Pneumonia"].astype(int)

# Error analysis
results = pd.DataFrame({
    "Lung Lesion FP": (pred_ll == 1) & (true_ll == 0),
    "Lung Lesion FN": (pred_ll == 0) & (true_ll == 1),
    "Pneumonia FP": (pred_pneu == 1) & (true_pneu == 0),
    "Pneumonia FN": (pred_pneu == 0) & (true_pneu == 1)
})

print(results.sum())

both_false_positive = (
    (pred_ll == 1) &
    (pred_pneu == 1) &
    (true_ll == 0) &
    (true_pneu == 0)
)

print("Both predicted positive, neither actually present:",
      both_false_positive.sum())

Lung Lesion FP    1217
Lung Lesion FN     483
Pneumonia FP      1383
Pneumonia FN       311
dtype: int64
Both predicted positive, neither actually present: 914


### Observation

The investigation of Lung Lesion and Pneumonia revealed a substantial overlap in false-positive predictions. Using a classification threshold of 0.7, the model produced 1217 false positives for Lung Lesion and 1383 for Pneumonia. Notably, 914 images were incorrectly classified as positive for both pathologies, despite neither being present in the ground truth.

## 5 Compare patient/image characteristics of the false positives

In [31]:

metadata_df = pd.DataFrame({
    "Age": XRAY_AGE_LIST,
    "Sex": XRAY_GENDER_LIST,
    "View Position": XRAY_VIEW_LIST
})


metadata_df["Both FP"] = np.asarray(both_fp)

metadata_df["Group"] = metadata_df["Both FP"].map({
    True: "Both false positives",
    False: "Other images"
})

In [32]:

view_counts = pd.crosstab(
    metadata_df["View Position"],
    metadata_df["Group"]
)

view_percentages = pd.crosstab(
    metadata_df["View Position"],
    metadata_df["Group"],
    normalize="index"
) * 100

In [33]:
fig = px.bar(
    view_percentages.reset_index().melt(
        id_vars="View Position",
        var_name="Group",
        value_name="Percentage"
    ),
    x="View Position",
    y="Percentage",
    color="Group",
    barmode="group",
    title="Simultaneous False Positives by X-ray View Position",
    text_auto=".2f"
)

fig.update_layout(
    yaxis_title="Percentage of images (%)",
    xaxis_title="View Position"
)

fig.show()

In [25]:
fig = px.box(
    metadata_df,
    x="Group",
    y="Age",
    color="Group",
    points="outliers",
    title="Patient Age: Both False Positives vs. Other Images"
)

fig.update_layout(
    xaxis_title="Image Group",
    yaxis_title="Patient Age"
)

fig.show()

In [28]:
sex_percentages = pd.crosstab(
    metadata_df["Sex"],
    metadata_df["Group"],
    normalize="columns"
) * 100

In [27]:
fig = px.bar(
    sex_percentages.reset_index().melt(
        id_vars="Sex",
        var_name="Group",
        value_name="Percentage"
    ),
    x="Sex",
    y="Percentage",
    color="Group",
    barmode="group",
    title="Sex Distribution: Both False Positives vs. Other Images",
    text_auto=".2f"
)

fig.update_layout(
    yaxis_title="Percentage (%)"
)

fig.show()

### Observation

Patient age: Older patients appear more frequently among images where the model incorrectly predicts both Lung Lesion and Pneumonia.

Image view: There is no apparent association between AP/PA view position and simultaneous false-positive predictions.

Further investigation of the image characteristics are done in research question 3.

Note: AI further suggested a A Mann–Whitney U test. We stopped here with research question 2.

## 6 Network visualizations

In [34]:
import networkx as nx

In [35]:
def show_similarity_network(correlation_matrix, threshold=0.8):

    # Create an undirected graph
    G = nx.Graph()

    # Add all pathologies as nodes
    pathologies = correlation_matrix.columns.tolist()
    G.add_nodes_from(pathologies)

    # Add edges based on correlation threshold
    for i, pathology1 in enumerate(pathologies):
        for j, pathology2 in enumerate(pathologies):

            if j <= i:
                continue

            correlation = correlation_matrix.loc[
                pathology1, pathology2
            ]

            if correlation >= threshold:
                G.add_edge(
                    pathology1,
                    pathology2,
                    weight=correlation
                )

    # Calculate node positions
    pos = nx.spring_layout(
        G,
        seed=42,
        weight="weight",
        k=1.5
    )

    # Create edge traces
    edge_traces = []

    for node1, node2, data in G.edges(data=True):

        x0, y0 = pos[node1]
        x1, y1 = pos[node2]

        correlation = data["weight"]

        edge_traces.append(
            go.Scatter(
                x=[x0, x1],
                y=[y0, y1],
                mode="lines",
                line=dict(
                    width=1 + 6 * (correlation - threshold) / (1 - threshold),
                    color="gray"
                ),
                hoverinfo="text",
                text=[
                    f"{node1} – {node2}: {correlation:.2f}",
                    f"{node1} – {node2}: {correlation:.2f}"
                ],
                showlegend=False
            )
        )

    # Calculate node degrees
    degrees = dict(G.degree())

    # Create node trace
    node_trace = go.Scatter(
        x=[pos[node][0] for node in G.nodes()],
        y=[pos[node][1] for node in G.nodes()],
        mode="markers+text",
        text=list(G.nodes()),
        textposition="top center",
        hovertext=[
            f"{node}<br>"
            f"Connections: {degrees[node]}"
            for node in G.nodes()
        ],
        hoverinfo="text",
        marker=dict(
            size=[
                18 + degrees[node] * 3
                for node in G.nodes()
            ],
            color=list(degrees.values()),
            colorscale="Viridis",
            showscale=True,
            colorbar=dict(title="Connections"),
            line=dict(width=1, color="black")
        )
    )

    # Create figure
    fig = go.Figure(
        data=edge_traces + [node_trace]
    )

    fig.update_layout(
        title=f"Pathology Similarity Network (r ≥ {threshold})",
        showlegend=False,
        width=1100,
        height=850,
        hovermode="closest",
        xaxis=dict(visible=False),
        yaxis=dict(visible=False),
        template="plotly_white"
    )

    fig.show()

    return G

In [37]:
G = show_similarity_network(
    correlation_matrix,
    threshold=0.7
)

### Observation

Pneumonia is strongly connected to other diseases, especially Lung Opacity. Generally they tend to form a cluster. 
Interpretation: the model has a tendency to act as a simple diseas detector, where several pathologies are predicted together.